# Regression Comparison Notebook

이 노트북은 `StatAutoLab` 앱의 회귀 결과를 직접 비교해보기 위한 용도입니다.

확인할 내용:
- 앱과 비슷한 전처리 + `LinearRegression` holdout 평가
- 전체 데이터 기준 OLS 요약
- `RandomForestRegressor`의 전형적인 결과 형식
- 왜 holdout 지표와 OLS 적합 결과가 다르게 보이는지 비교


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.evaluate import evaluate_regression
from app.preprocessing import build_preprocessing_pipeline

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'examples' / 'regression_sample.csv'
TARGET = 'spending_score'
FEATURES = ['age', 'income', 'city', 'visits']
TEST_SIZE = 0.2
RANDOM_STATE = 42

print('DATA_PATH =', DATA_PATH)
print('exists =', DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)
df

## 1. 앱과 비슷한 holdout 평가: LinearRegression

이 셀은 `run_analysis.py`의 회귀 평가와 비슷하게,
- target 결측 제거
- 선택한 feature만 사용
- 학습/검증 분리
- `LinearRegression` 학습
- RMSE, MAE, R2 계산
을 수행합니다.


In [ ]:
model_df = df.dropna(subset=[TARGET]).copy()
preprocessor, features_df, summary = build_preprocessing_pipeline(
    model_df,
    TARGET,
    feature_columns=FEATURES,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    features_df,
    model_df[TARGET],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

linear_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ]
)
linear_pipeline.fit(X_train, y_train)
linear_valid_predictions = linear_pipeline.predict(X_valid)
linear_holdout_metrics = evaluate_regression(y_valid, linear_valid_predictions)

print('Selected features:', summary.selected_feature_columns)
print('LinearRegression holdout metrics:', linear_holdout_metrics)

linear_comparison_df = pd.DataFrame({
    'actual': y_valid.reset_index(drop=True),
    'predicted': pd.Series(linear_valid_predictions),
})
linear_comparison_df['residual'] = linear_comparison_df['actual'] - linear_comparison_df['predicted']
linear_comparison_df

## 2. 전체 데이터 기준 OLS 요약

이 셀은 앱 대시보드의 OLS 요약 블록과 비슷하게, 전체 데이터에 적합한 선형회귀 결과를 `statsmodels`로 보여줍니다.

주의:
- 이 결과는 holdout 검증 성능이 아닙니다.
- 전체 데이터에 fit한 결과라서 보통 더 좋아 보입니다.


In [ ]:
full_preprocessor, full_features_df, _ = build_preprocessing_pipeline(
    model_df,
    TARGET,
    feature_columns=FEATURES,
)

X_transformed = full_preprocessor.fit_transform(full_features_df)
if hasattr(X_transformed, 'toarray'):
    X_transformed = X_transformed.toarray()

feature_names = full_preprocessor.get_feature_names_out()
feature_names = [name.split('__', 1)[1] if '__' in name else name for name in feature_names]

X_ols = pd.DataFrame(X_transformed, columns=feature_names, index=model_df.index)
X_ols = sm.add_constant(X_ols, has_constant='add')
ols_result = sm.OLS(model_df[TARGET].astype(float), X_ols.astype(float)).fit()

print(ols_result.summary())

## 3. RandomForestRegressor는 보통 어떻게 보나?

랜덤포레스트 회귀는 OLS처럼 p-value나 회귀계수표가 나오지 않습니다.
보통 아래를 봅니다.
- RMSE, MAE, R2
- 실제값 vs 예측값
- 변수 중요도 (feature importance)


In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    ]
)
rf_pipeline.fit(X_train, y_train)
rf_valid_predictions = rf_pipeline.predict(X_valid)
rf_holdout_metrics = evaluate_regression(y_valid, rf_valid_predictions)

print('RandomForestRegressor holdout metrics:', rf_holdout_metrics)

rf_comparison_df = pd.DataFrame({
    'actual': y_valid.reset_index(drop=True),
    'predicted': pd.Series(rf_valid_predictions),
})
rf_comparison_df['residual'] = rf_comparison_df['actual'] - rf_comparison_df['predicted']
rf_comparison_df

In [ ]:
rf_model = rf_pipeline.named_steps['model']
rf_preprocessor = rf_pipeline.named_steps['preprocessor']
rf_feature_names = rf_preprocessor.get_feature_names_out()
rf_feature_names = [name.split('__', 1)[1] if '__' in name else name for name in rf_feature_names]

rf_importance_df = pd.DataFrame({
    'feature': rf_feature_names,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)
rf_importance_df

## 4. 전체 데이터 적합도와 holdout 성능 비교

왜 대시보드 값과 holdout 값이 달라 보이는지 직접 비교합니다.


In [ ]:
full_fit_pipeline = Pipeline(
    steps=[
        ('preprocessor', full_preprocessor),
        ('model', LinearRegression()),
    ]
)
full_fit_pipeline.fit(full_features_df, model_df[TARGET])
full_fit_predictions = full_fit_pipeline.predict(full_features_df)
full_fit_metrics = evaluate_regression(model_df[TARGET], full_fit_predictions)

pd.DataFrame(
    [
        {'evaluation_basis': 'linear_holdout_validation', **linear_holdout_metrics},
        {'evaluation_basis': 'rf_holdout_validation', **rf_holdout_metrics},
        {'evaluation_basis': 'linear_full_data_fit', **full_fit_metrics},
    ]
)

## 5. 해석 메모

- `LinearRegression + OLS`: 회귀계수, p-value, 신뢰구간 같은 전형적인 회귀분석 해석 가능
- `RandomForestRegressor`: 예측 성능과 변수 중요도 중심으로 해석
- 표본이 8행뿐이라 결과가 매우 쉽게 흔들릴 수 있음
- 따라서 이 데이터로는 `모델이 아주 좋다`고 단정하기 어렵고 `예시용` 또는 `코드 검증용`에 가깝습니다.
